Create a new dataloader to load up soundscape data at 5 second intervals.
We embed them as we process the sequence.
These are aligned with the perch outputs.
We see how well we can predict the perch ouputs via cross-entropy.

In [ ]:
import polars as pl
from pathlib import Path
from birdclef.mel2vec.loaders import get_index, get_word_vectors
import numpy as np
import tqdm
import multiprocessing as mp

In [2]:
scratch = Path("~/scratch/birdclef/2025").expanduser()
root = scratch / "mel2vec-v1"

centroid_path = list((root / "tokenizer").glob("**/centroids.npy"))[0].as_posix()
wordvector_path = list((root / "word2vec").glob("**/epochs=100/word2vec.wordvectors"))[
    0
].as_posix()

index = get_index(centroid_path)
wordvectors = get_word_vectors(wordvector_path)
index, wordvectors

(<faiss.swigfaiss_avx512.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7fffe411a670> >,
 <gensim.models.keyedvectors.KeyedVectors at 0x7fffe4119de0>)

In [3]:
mfcc_df = pl.scan_parquet(f"{scratch}/mfcc-soundscape/data").sort("file", "timestamp")
mfcc_df.collect_schema(), mfcc_df.count().collect()

(Schema([('index', Int64),
         ('file', String),
         ('timestamp', Float64),
         ('mfcc', List(Float32)),
         ('part', Int64)]),
 shape: (1, 5)
 ┌─────────┬─────────┬───────────┬─────────┬─────────┐
 │ index   ┆ file    ┆ timestamp ┆ mfcc    ┆ part    │
 │ ---     ┆ ---     ┆ ---       ┆ ---     ┆ ---     │
 │ u32     ┆ u32     ┆ u32       ┆ u32     ┆ u32     │
 ╞═════════╪═════════╪═══════════╪═════════╪═════════╡
 │ 4658754 ┆ 4658754 ┆ 4658754   ┆ 4658754 ┆ 4658754 │
 └─────────┴─────────┴───────────┴─────────┴─────────┘)

Then let's figure out how to assign them in such a way that they are aligned to 5 second intervals.

In [ ]:
# assign a start time based on the closest 5-second interval
# then group by the start time, and create an array ordered by timestamp
def get_token(mfcc):
    X = np.array(mfcc).reshape(1, -1)
    _, token = index.search(X, 1)
    return int(token[0][0])


test_df = mfcc_df.filter(pl.col("part") == 0).sort("file", "timestamp")
mfcc_series = test_df.select("mfcc").collect().get_column("mfcc")
with mp.Pool(8) as pool:
    tokens = pool.map(get_token, tqdm.tqdm(mfcc_series, desc="Tokenizing MFCCs"))

tokenized_test = (
    test_df.with_columns(
        (pl.col("timestamp") // 5 * 5).alias("start_time"),
        pl.Series(tokens).alias("token"),
        pl.col("file").str.split("/").list.last().alias("file"),
    )
    .group_by("file", "start_time")
    .agg(pl.col("token").sort_by("timestamp").alias("tokens"))
)
tokenized_test.head().collect()

Tokenizing MFCCs: 100%|██████████| 46942/46942 [00:01<00:00, 42565.88it/s]


file,start_time,tokens
str,f64,list[i64]
"""H73_20230430_114000.ogg""",50.0,"[14970, 15829, … 13880]"
"""H95_20230510_152500.ogg""",30.0,"[13743, 7476, … 9987]"
"""H18_20230522_043000.ogg""",20.0,"[8909, 1766, … 6514]"
"""H11_20230426_235500.ogg""",5.0,"[5195, 12646, … 9059]"
"""H73_20230430_114000.ogg""",15.0,"[11477, 992, … 2593]"


In [13]:
shared = Path("~/shared/birdclef").expanduser()
perch = pl.scan_parquet(f"{shared}/2025/infer-soundscape/Perch/parts/predict/*.parquet")
columns = perch.collect_schema().names()
perch = perch.select(
    pl.col("file").str.split("/").list.last().alias("file"),
    "start_time",
    "end_time",
    (pl.concat_list(columns[3:]).list.to_array(len(columns[3:])).alias("logits")),
).sort("file", "start_time")
perch.select(pl.len()).collect().item()

116712

In [15]:
perch.collect_schema()

Schema([('file', String),
        ('start_time', Float64),
        ('end_time', Float64),
        ('logits', Array(Float64, shape=(10932,)))])

In [16]:
# get the token dataset now
def get_token(mfcc):
    X = np.array(mfcc).reshape(1, -1)
    _, token = index.search(X, 1)
    return int(token[0][0])


mfcc_series = mfcc_df.select("mfcc").collect().get_column("mfcc")
with mp.Pool(8) as pool:
    tokens = pool.map(get_token, tqdm.tqdm(mfcc_series, desc="Tokenizing MFCCs"))

tokenized = (
    mfcc_df.with_columns(
        (pl.col("timestamp") // 5 * 5).alias("start_time"),
        pl.Series(tokens).alias("token"),
        pl.col("file").str.split("/").list.last().alias("file"),
        "part",
    )
    .group_by("part", "file", "start_time")
    .agg(pl.col("token").sort_by("timestamp").alias("tokens"))
)
tokenized.select(pl.len()).collect().item()

Tokenizing MFCCs: 100%|██████████| 4658754/4658754 [01:48<00:00, 43108.02it/s] 


116712

In [20]:
tokenized.join(
    perch,
    on=["file", "start_time"],
    how="left",
).select(
    "part",
    "file",
    "start_time",
    "tokens",
    "logits",
).collect().write_parquet(
    f"{scratch}/soundscape-token-perch", use_pyarrow=True, partition_by=["part"]
)